# 🧠 Prediction-of-Prediction (PoP) — Full PoC

**Concept:** A meta-learning layer that predicts when the base LLM is wrong, BEFORE trusting its output.

```
DistilGPT-2 (base model)
       ↓
  logits + probability distribution
       ↓
  PoP Layer (neural net) → "this distribution pattern = likely error"
       ↓
  Safety Guard → only correct if PoP is confident
       ↓
  Final output (either LLM or corrected)
```

**How PoP learns:**
- LLM makes a prediction (token + probability distribution)
- We know the correct answer
- PoP learns: "when distribution looks like THIS → LLM is probably wrong"
- Next time, PoP flags bad predictions before we trust them

## 1. Install Dependencies

In [ ]:
!pip install transformers torch scikit-learn numpy -q

## 2. Load Base LLM (DistilGPT-2)

In [ ]:
import sys
sys.path.insert(0, '/content/pop-repo')

from pop.core.llm_base import create_llm

# Load DistilGPT-2 — 82M params, runs on CPU
llm = create_llm('distilgpt2', device='cpu')
print(f'Model loaded: {llm.model_name} | Vocab: {llm.vocab_size} | Device: {llm.device}')

## 3. Test Base LLM — See What It Predicts

In [ ]:
# See what DistilGPT-2 thinks
test_prompts = [
    'The capital of France is',
    'The chemical symbol for gold is',
    'The largest planet is',
    '2 + 2 equals',
    'The opposite of hot is',
]

print('='*60)
print('BASE LLM PREDICTIONS (DistilGPT-2, no PoP)')
print('='*60)

for prompt in test_prompts:
    result = llm.predict_next_token(prompt, top_k=5)
    print(f'\n"{prompt}"')
    for i, (token, prob) in enumerate(zip(result['top_tokens'], result['top_probs'])):
        marker = ' ← TOP' if i == 0 else ''
        print(f'  {i+1}. "{token}" ({prob:.4f}){marker}')

## 4. Create PoP Layer + Debugger

- **PoP Layer:** Neural net that extracts 16 features from the probability distribution and predicts error magnitude, confidence, and direction
- **Debugger:** Logs every prediction for analysis

In [ ]:
import torch
from pop.core.pop_layer_llm import create_pop_llm
from pop.core.debugger import PoPDebugger

# Create untrained PoP layer
pop = create_pop_llm(vocab_size=llm.vocab_size, device='cpu')
debugger = PoPDebugger(verbose=True)

print(f'PoP layer created: vocab={pop.vocab_size}, trained={pop.is_trained}')

## 5. Baseline — Untrained PoP

Before training, PoP has random weights. Let's see what it does.

In [ ]:
print('\n' + '='*60)
print('BASELINE: Untrained PoP (random predictions)')
print('='*60)

for prompt in test_prompts:
    logits = llm.get_logits(prompt)
    probs = torch.softmax(logits, dim=-1)
    result = pop.predict(logits.unsqueeze(0), probs.unsqueeze(0))
    
    llm_result = llm.predict_next_token(prompt, top_k=1)
    print(f'\n"{prompt}" → LLM says: "{llm_result["top_tokens"][0]}"')
    print(f'  PoP: error={result["error_magnitude"]:.3f} | conf={result["confidence"]:.3f} | correct={result["should_correct"]}')

## 6. TRAIN PoP on Real Data

This is the core loop:
1. Feed prompt to LLM → get logits + probs
2. Check if LLM's top prediction matches known answer
3. If wrong → error_magnitude=1.0 (LLM made a mistake)
4. If right → error_magnitude=0.0 (LLM was correct)
5. PoP learns: this distribution pattern → this error level

In [ ]:
from pop.core.training_data import get_all_facts

facts = get_all_facts()
print(f'Training examples: {len(facts)}')

# First, let's see how many DistilGPT-2 gets wrong (these are the valuable examples)
wrong_count = 0
for f in facts:
    result = llm.predict_next_token(f['prompt'], top_k=1)
    if result['top_tokens'][0].strip().lower() != f['answer'].strip().lower():
        wrong_count += 1

print(f'DistilGPT-2 errors: {wrong_count}/{len(facts)} ({wrong_count/len(facts):.0%})')

In [ ]:
# TRAIN PoP — 5 epochs over all facts
import logging
logging.basicConfig(level=logging.INFO)

EPOCHS = 5
training_history = []

print('\n' + '='*60)
print('TRAINING PoP LAYER')
print('='*60)

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    errors_found = 0
    
    for f in facts:
        prompt = f['prompt']
        correct = f['answer']
        
        # Get LLM outputs
        logits = llm.get_logits(prompt)
        probs = torch.softmax(logits, dim=-1)
        llm_result = llm.predict_next_token(prompt, top_k=1)
        
        predicted = llm_result['top_tokens'][0]
        pred_prob = llm_result['top_probs'][0]
        
        # Is LLM wrong?
        is_wrong = predicted.strip().lower() != correct.strip().lower()
        
        # Correct token probability
        if correct in llm_result.get('top_tokens', []):
            idx = llm_result['top_tokens'].index(correct)
            correct_prob = llm_result['top_probs'][idx]
        else:
            full_probs = probs[0].cpu().numpy()
            ids = llm.tokenizer.encode(correct)
            correct_prob = float(full_probs[ids[0]]) if ids else 0.0
        
        # Error labels
        error_mag = 1.0 if is_wrong else 0.0
        conf = pred_prob
        error_dir = (pred_prob - correct_prob) if is_wrong else 0.0
        
        # Train
        loss = pop.train_step(logits.unsqueeze(0), probs.unsqueeze(0), error_mag, conf, error_dir)
        epoch_loss += loss['loss']
        if is_wrong:
            errors_found += 1
    
    avg_loss = epoch_loss / len(facts)
    training_history.append({'epoch': epoch+1, 'loss': avg_loss, 'errors': errors_found})
    print(f'  Epoch {epoch+1}/{EPOCHS} — loss: {avg_loss:.4f} | LLM errors seen: {errors_found}/{len(facts)}')

print('\n✅ Training complete!')

## 7. Test TRAINED PoP — Full System with Debugger

In [ ]:
from pop.core.integration import create_pop_system

# We can't re-create the system (it reloads LLM), so manually wire trained PoP
# into a fresh system. Or just test directly:

print('\n' + '='*60)
print('TRAINED PoP — Full Predictions with Debugger')
print('='*60)

test_cases = [
    {'prompt': 'The capital of France is', 'answer': ' Paris'},
    {'prompt': 'The chemical symbol for gold is', 'answer': ' Au'},
    {'prompt': 'The largest planet is', 'answer': ' Jupiter'},
    {'prompt': '2 + 2 equals', 'answer': ' 4'},
    {'prompt': 'The opposite of hot is', 'answer': ' cold'},
    {'prompt': 'The speed of light is approximately', 'answer': ' 3'},
    {'prompt': 'The mitochondria is the', 'answer': ' powerhouse'},
    {'prompt': 'Shakespeare wrote', 'answer': ' Hamlet'},
    {'prompt': 'The Mona Lisa was painted by', 'answer': ' Leonardo'},
    {'prompt': 'The atomic number of carbon is', 'answer': ' 6'},
    {'prompt': 'World War II ended in', 'answer': ' 1945'},
    {'prompt': 'The first man on the moon was', 'answer': ' Neil'},
    {'prompt': 'The pyramids are in', 'answer': ' Egypt'},
    {'prompt': 'Newton\'s first law is about', 'answer': ' inertia'},
    {'prompt': 'The square root of 144 is', 'answer': ' 12'},
]

debugger = PoPDebugger(verbose=True)

for tc in test_cases:
    prompt = tc['prompt']
    correct = tc['answer']
    
    # LLM prediction
    llm_result = llm.predict_next_token(prompt, top_k=5)
    top_token = llm_result['top_tokens'][0]
    top_prob = llm_result['top_probs'][0]
    
    # PoP analysis
    logits = llm.get_logits(prompt)
    probs = torch.softmax(logits, dim=-1)
    pop_result = pop.predict(logits.unsqueeze(0), probs.unsqueeze(0))
    
    # Safety guard
    should_correct = pop_result['confidence'] > 0.7 and pop_result['error_magnitude'] > 0.3
    
    if should_correct:
        final_token = llm_result['top_tokens'][1] if len(llm_result['top_tokens']) > 1 else top_token
        final_prob = llm_result['top_probs'][1] if len(llm_result['top_probs']) > 1 else top_prob
        decision = 'CORRECTED'
    else:
        final_token = top_token
        final_prob = top_prob
        decision = 'TRUST_LLM'
    
    debugger.log_prediction(
        input_text=prompt,
        llm_token=top_token,
        llm_prob=top_prob,
        llm_top5=[{'token': t, 'prob': p} for t, p in zip(llm_result['top_tokens'][:5], llm_result['top_probs'][:5])],
        pop_error_magnitude=pop_result['error_magnitude'],
        pop_confidence=pop_result['confidence'],
        pop_direction=pop_result['error_direction'],
        decision=decision,
        final_token=final_token,
        final_prob=final_prob,
        correct_token=correct
    )

## 8. Debugger Summary

In [ ]:
debugger.print_summary()

## 9. Error Distribution — See PoP's Decision Boundary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

errors = debugger.get_error_distribution()

plt.figure(figsize=(10, 4))
plt.hist(errors, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(x=0.3, color='red', linestyle='--', label='Correction threshold (0.3)')
plt.xlabel('PoP Error Magnitude')
plt.ylabel('Count')
plt.title('Distribution of PoP Error Predictions')
plt.legend()
plt.show()

print(f'Mean error magnitude: {np.mean(errors):.4f}')
print(f'Std: {np.std(errors):.4f}')
print(f'Max: {np.max(errors):.4f}')

## 10. Missed Corrections & False Corrections

In [ ]:
# Cases where PoP should have corrected but didn't
missed = debugger.get_missed_corrections()
print(f'\n MISSED CORRECTIONS ({len(missed)}):')
for e in missed:
    print(f'  "{e.input_text}" → LLM said "{e.llm_token}" but answer was "{e.correct_token}"')

# Cases where PoP corrected but made it worse
bad = debugger.get_false_corrections()
print(f'\n FALSE CORRECTIONS ({len(bad)}):')
for e in bad:
    print(f'  "{e.input_text}" → LLM had "{e.llm_token}" (correct!) but PoP changed to "{e.final_token}"')

## 11. Retrain on Missed Cases — Iterative Improvement

PoP can learn from its mistakes. Feed the missed cases back in.

In [ ]:
# Retrain specifically on cases PoP got wrong
retrain_examples = []
for e in missed:
    retrain_examples.append({'prompt': e.input_text, 'answer': e.correct_token})

if retrain_examples:
    print(f'Retraining on {len(retrain_examples)} missed cases...')
    for epoch in range(3):
        for ex in retrain_examples:
            logits = llm.get_logits(ex['prompt'])
            probs = torch.softmax(logits, dim=-1)
            llm_result = llm.predict_next_token(ex['prompt'], top_k=1)
            
            predicted = llm_result['top_tokens'][0]
            pred_prob = llm_result['top_probs'][0]
            correct = ex['answer']
            
            is_wrong = predicted.strip().lower() != correct.strip().lower()
            error_mag = 1.0 if is_wrong else 0.0
            
            pop.train_step(logits.unsqueeze(0), probs.unsqueeze(0), error_mag, pred_prob, error_mag * 0.5)
    print('✅ Retraining complete!')
else:
    print('No missed cases to retrain on!')

## 12. Save Debug Log

In [ ]:
debugger.to_json('/content/pop_debug_log.json')
print('Debug log saved to /content/pop_debug_log.json')

## Summary

### What we built:
1. **Base LLM** (DistilGPT-2) — makes predictions, gives us probability distributions
2. **PoP Layer** — neural net that extracts 16 features from the distribution, predicts if LLM is wrong
3. **Safety Guard** — only applies correction if PoP is confident AND error magnitude is high
4. **Debugger** — full observability: logs every prediction, tracks accuracy, finds missed/false corrections
5. **Training Pipeline** — feeds (distribution → error) pairs to PoP, learns iteratively

### How it works:
```
DistilGPT-2 → logits/probs → PoP analyzes distribution pattern
                                     ↓
                            "this pattern = likely error"
                                     ↓
                         Safety Guard checks confidence
                                     ↓
                    Trust LLM  OR  Pick alternative token
```

### Next steps:
- Larger training datasets (HellaSwag, MMLU)
- Conformal prediction for statistical guarantees
- Multi-token lookahead
- Calibration metrics (ECE)